In [1]:
import os
import getpass

In [2]:
os.environ['HF_TOKEN'] = getpass.getpass('Your hugginface token:')

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "swap-uniba/LLaMAntino-2-7b-hf-dolly-ITA"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)


pytorch_model.bin.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

In [5]:
import torch 

model  = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="auto").eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [6]:
ALPACA = ("Di seguito è riportata un'istruzione che descrive un'attività, "
          "accompagnata da un input che aggiunge ulteriore informazione. "
          "Scrivi una risposta che completi adeguatamente la richiesta.\n\n"
          "### Istruzione:\n{instruction}\n\n"
          "### Input:\n{input}\n\n"
          "### Risposta:\n")

@torch.no_grad()
def llm_complete(instruction, input_text, max_new_tokens=200):
    prompt    = ALPACA.format(instruction=instruction, input=input_text)
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    out = model.generate(input_ids=input_ids, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)

In [7]:
import json, re

ISTRUZIONE = ("Ti viene fornita una frase italiana tokenizzata e numerata. "
              "Restituisci gli indici dei token complessi (parole rare o costruzioni "
              "sintattiche difficili) da rimuovere o sostituire per semplificarla. "
              'Rispondi solo con un JSON nel formato {"complex": [indici]}.')

def _format_indexed(tokens):
    return "\n".join(f"{i}: {t}" for i, t in enumerate(tokens))

def _parse_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(m.group(0)) if m else {"complex": []}

In [8]:
def predict_llm(eval_set, return_motivi=False):
    preds, motivi_all = [], []
    for ex in eval_set:
        n = len(ex["tokens"])
        try:
            resp = llm_complete(ISTRUZIONE, _format_indexed(ex["tokens"]))
            data = _parse_json(resp)
            idxs = [i for i in data.get("complex", []) if isinstance(i, int) and 0 <= i < n]
            motivi_all.append(data.get("motivi", {}))
        except Exception:
            idxs = []; motivi_all.append({})
        p = [0]*n
        for i in idxs: p[i] = 1
        preds.append(p)
    return (preds, motivi_all) if return_motivi else preds

In [9]:
import numpy as np
import pandas as pd
import torch
import spacy
from difflib import SequenceMatcher
from IPython.display import display, HTML
from captum.attr import LayerIntegratedGradients
from captum.attr import visualization as viz
from transformers import AutoModelForSequenceClassification, AutoTokenizer

nlp = spacy.load("it_core_news_md")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [10]:
test = pd.read_parquet("/code/HLTproject_code/data/test.parquet")

orig = (test[test["is_original"]]
        .drop_duplicates("original_sentence_idx")
        .set_index("original_sentence_idx")["text"])
simp = (test[~test["is_original"]]
        .drop_duplicates("original_sentence_idx")
        .set_index("original_sentence_idx")["text"])

pairs = (pd.concat([orig.rename("original"), simp.rename("simplification")], axis=1)
           .dropna().reset_index())
print(f"Coppie nel test set: {len(pairs)}")

def silver_labels_tipate(original, simplification):
    doc_o = [t for t in nlp(original) if not t.is_space]
    lem_o = [t.lemma_.lower() for t in doc_o]
    lem_s = [t.lemma_.lower() for t in nlp(simplification) if not t.is_space]
    tipo = ["keep"] * len(doc_o)
    for tag, i1, i2, j1, j2 in SequenceMatcher(a=lem_o, b=lem_s, autojunk=False).get_opcodes():
        if tag == "delete":
            for i in range(i1, i2): tipo[i] = "del"
        elif tag == "replace":
            for i in range(i1, i2): tipo[i] = "sub"
    tokens = [t.text for t in doc_o]
    labels = [int(x != "keep") for x in tipo]
    return tokens, labels, tipo

eval_set = []
for _, row in pairs.iterrows():
    doc_o = [t for t in nlp(row["original"]) if not t.is_space]
    tokens, labels, tipo = silver_labels_tipate(row["original"], row["simplification"])
    eval_set.append({
        "idx":        row["original_sentence_idx"],
        "original":   row["original"],
        "tokens":     tokens,
        "silver":     labels,
        "tipo":       tipo,
        "simp_tokens": [t.text for t in nlp(row["simplification"]) if not t.is_space],
    })

print(f"Eval set pronto: {len(eval_set)} frasi")

Coppie nel test set: 7539
Eval set pronto: 7539 frasi


In [14]:
import numpy as np

rng = np.random.default_rng(42)
N = min(1, len(eval_set))
sample_idx  = rng.choice(len(eval_set), size=N, replace=False)
eval_sample = [eval_set[i] for i in sample_idx]

print(f"eval_sample: {len(eval_sample)} frasi (su {len(eval_set)} totali)")

eval_sample: 1 frasi (su 7539 totali)


In [15]:
preds_llm = predict_llm(eval_sample)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [16]:
preds_llm

[[0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]]

In [17]:
i = 0
toks = eval_sample[i]["tokens"]
pred = preds_llm[i]
complesse = [t for t, p in zip(toks, pred) if p == 1]
print("Frase   :", " ".join(toks))
print("Complesse:", complesse)

Frase   : Nel primo giorno il tempo migliore è fatto da Felipe Massa su Ferrari in 1'40"170 , tempo di circa sette decimi più alto di quello della " pole " fatto segnare la settimana precedente .
Complesse: ['primo', 'giorno', 'il', 'tempo', 'migliore', 'è', 'fatto', 'da', 'Felipe', 'Massa', 'su', 'Ferrari', 'in', '1\'40"170', ',', 'tempo', 'di', 'circa', 'sette', 'decimi', 'più', 'alto', 'di', 'quello', 'della', '"', 'pole', '"', 'fatto', 'segnare', 'la', 'settimana', 'precedente', '.']


In [18]:
def confronta(ex, pred):
    toks, silver = ex["tokens"], ex["silver"]
    print("LLM    :", " ".join(f"[{t}]" if p else t for t, p in zip(toks, pred)))
    print("SILVER :", " ".join(f"[{t}]" if s else t for t, s in zip(toks, silver)))

confronta(eval_sample[0], preds_llm[0])

LLM    : Nel [primo] [giorno] [il] [tempo] [migliore] [è] [fatto] [da] [Felipe] [Massa] [su] [Ferrari] [in] [1'40"170] [,] [tempo] [di] [circa] [sette] [decimi] [più] [alto] [di] [quello] [della] ["] [pole] ["] [fatto] [segnare] [la] [settimana] [precedente] [.]
SILVER : [Nel] [primo] [giorno] [il] [tempo] [migliore] [è] [fatto] [da] Felipe Massa [su] [Ferrari] in 1'40"170 [,] [tempo] [di] [circa] [sette] [decimi] [più] [alto] [di] [quello] [della] ["] [pole] ["] [fatto] [segnare] [la] [settimana] [precedente] .


In [21]:
from IPython.display import HTML, display
def mostra_html(toks, pred):
    display(HTML(" ".join(
        f'<span style="background:#fd9">{t}</span>' if p else t
        for t, p in zip(toks, pred))))

mostra_html(eval_sample[0]["tokens"], preds_llm[0])

In [22]:
def record(i):
    ex = eval_sample[i]
    print("ORIGINALE   :", " ".join(ex["tokens"]))
    print("SEMPLIFICATA:", " ".join(ex["simp_tokens"]))
    print("LLM         :", " ".join(f"[{t}]" if p else t
                                     for t, p in zip(ex["tokens"], preds_llm[i])))
    print("SILVER      :", " ".join(f"[{t}]" if s else t
                                     for t, s in zip(ex["tokens"], ex["silver"])))

record(0)

ORIGINALE   : Nel primo giorno il tempo migliore è fatto da Felipe Massa su Ferrari in 1'40"170 , tempo di circa sette decimi più alto di quello della " pole " fatto segnare la settimana precedente .
SEMPLIFICATA: Felipe Massa guidò la prima sessione di prova del Gran Premio d' Australia 2009 in 1'40"170 .
LLM         : Nel [primo] [giorno] [il] [tempo] [migliore] [è] [fatto] [da] [Felipe] [Massa] [su] [Ferrari] [in] [1'40"170] [,] [tempo] [di] [circa] [sette] [decimi] [più] [alto] [di] [quello] [della] ["] [pole] ["] [fatto] [segnare] [la] [settimana] [precedente] [.]
SILVER      : [Nel] [primo] [giorno] [il] [tempo] [migliore] [è] [fatto] [da] Felipe Massa [su] [Ferrari] in 1'40"170 [,] [tempo] [di] [circa] [sette] [decimi] [più] [alto] [di] [quello] [della] ["] [pole] ["] [fatto] [segnare] [la] [settimana] [precedente] .


In [23]:
import pickle
with open("eval_set_full.pkl", "wb") as f:
    pickle.dump(eval_set, f)

In [24]:
import pickle
with open("eval_set_venti.pkl", "wb") as f:
    pickle.dump(eval_set[:20], f)